<a href="https://colab.research.google.com/github/cameronliddle/ThesisAIDetection/blob/main/DenseNet121.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# Install and Mount
!pip install timm
from google.colab import drive
drive.mount('/content/drive')
!cp -r /content/drive/MyDrive/Thesis/Processed_Dataset/resplit_dataset /content/
#  Imports
import os
import random
import shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

# Model Loader
import timm

def load_model_architecture(model_name, input_shape):
    if model_name == 'densenet121':
        base_model = tf.keras.applications.DenseNet121(
            input_shape=input_shape,
            include_top=False,
            weights='imagenet'
        )
    else:
        raise ValueError(f"Unknown model name: {model_name}")
    return base_model

# Paths
base_dir = '/content/resplit_dataset'
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

save_dir = '/content/drive/MyDrive/Thesis/Processed_Dataset/saved_models'
os.makedirs(save_dir, exist_ok=True)

model_name = 'densenet121'
input_shape = (224, 224, 3)

model_save_path = os.path.join(save_dir, f'{model_name}_full_dataset.h5')
results_save_path = os.path.join(save_dir, f'{model_name}_results.json')
conf_matrix_save_path = os.path.join(save_dir, f'{model_name}_confusion_matrix.png')

# Data Generators
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir, target_size=input_shape[:2], batch_size=batch_size, class_mode='binary'
)
val_generator = val_test_datagen.flow_from_directory(
    val_dir, target_size=input_shape[:2], batch_size=batch_size, class_mode='binary', shuffle=False
)
test_generator = val_test_datagen.flow_from_directory(
    test_dir, target_size=input_shape[:2], batch_size=batch_size, class_mode='binary', shuffle=False
)

# Build Model
base_model = load_model_architecture(model_name, input_shape)

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dense(128, activation='relu')(x)
output = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs=base_model.input, outputs=output)

# Compile Model
for layer in base_model.layers:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(" Model compiled!")

# Callbacks
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
checkpoint = tf.keras.callbacks.ModelCheckpoint(model_save_path, monitor='val_loss', save_best_only=True, verbose=1)

# Train Phase 1
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

# Fine-tuning Phase (Unfreeze)
for layer in base_model.layers:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

fine_tune_history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=[early_stop, checkpoint],
    verbose=1
)

model.save(model_save_path)
print(f" Final model saved to: {model_save_path}")

#  Evaluate and Save Metrics
val_loss, val_accuracy = model.evaluate(val_generator, verbose=1)
y_val_true = val_generator.classes
y_val_pred_prob = model.predict(val_generator, verbose=1)
y_val_pred = (y_val_pred_prob > 0.5).astype(int).flatten()

val_f1 = f1_score(y_val_true, y_val_pred)
val_precision = precision_score(y_val_true, y_val_pred)
val_recall = recall_score(y_val_true, y_val_pred)

test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
y_test_true = test_generator.classes
y_test_pred_prob = model.predict(test_generator, verbose=1)
y_test_pred = (y_test_pred_prob > 0.5).astype(int).flatten()

test_f1 = f1_score(y_test_true, y_test_pred)
test_precision = precision_score(y_test_true, y_test_pred)
test_recall = recall_score(y_test_true, y_test_pred)

# Save Results
results = {
    'Validation Accuracy (%)': round(val_accuracy * 100, 2),
    'Validation Loss': round(val_loss, 4),
    'Validation F1-Score': round(val_f1, 4),
    'Validation Precision': round(val_precision, 4),
    'Validation Recall': round(val_recall, 4),
    'Test Accuracy (%)': round(test_accuracy * 100, 2),
    'Test Loss': round(test_loss, 4),
    'Test F1-Score': round(test_f1, 4),
    'Test Precision': round(test_precision, 4),
    'Test Recall': round(test_recall, 4)
}

with open(results_save_path, 'w') as f:
    json.dump(results, f, indent=4)

print(f" Results saved to {results_save_path}")

conf_matrix = confusion_matrix(y_test_true, y_test_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('DenseNet121 Confusion Matrix (Test Set)')
plt.savefig(conf_matrix_save_path)
plt.show()

print(f" Confusion matrix saved to {conf_matrix_save_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 12595 images belonging to 2 classes.
Found 3001 images belonging to 2 classes.
Found 3006 images belonging to 2 classes.
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
 Model compiled!


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30


FailedPreconditionError: Graph execution error:

Detected at node StatefulPartitionedCall defined at (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main

  File "<frozen runpy>", line 88, in _run_code

  File "/usr/local/lib/python3.11/dist-packages/colab_kernel_launcher.py", line 37, in <module>

  File "/usr/local/lib/python3.11/dist-packages/traitlets/config/application.py", line 992, in launch_instance

  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelapp.py", line 712, in start

  File "/usr/local/lib/python3.11/dist-packages/tornado/platform/asyncio.py", line 205, in start

  File "/usr/lib/python3.11/asyncio/base_events.py", line 608, in run_forever

  File "/usr/lib/python3.11/asyncio/base_events.py", line 1936, in _run_once

  File "/usr/lib/python3.11/asyncio/events.py", line 84, in _run

  File "/usr/local/lib/python3.11/dist-packages/ipykernel/kernelbase.py", line 510, in dispatch_queue

  File "/usr/local/lib/python3.11/dist-pa